# ![icon](./images/uva-icon-57x57.png) WEEK 12 Snowflake CORTEX

| **Working Efficiently With Software** |
| :--: |
| **Data Engineering** |
| **School of Data Science** |
| **University of Virginia** |

# ![icon](./images/uva-icon-57x57.png) Announcements & Agenda

## T-Shirt sizing the remaining labs.
- ~~Setting up snowflake connector.  **M/L**~~
- ~~Dockerizing our pipeline.~~  **XL**
- ~~Git repo in snowflake.~~  **M**
- ~~DBT-expectations~~ **M**   

## We'll probably do this lab "in class" on the 4th.
- Snowflake CORTEX.  **S**  **<===== Snowflake  ===**

## Today
* Walk through part of the CORTEX LAB
* Quick review of what we did in class


# CORTEX LAB

# ![icon](./images/uva-icon-57x57.png) Step 0

**NB:**
```sql
SET MY_SCHEMA = (SELECT CURRENT_USER());
USE SCHEMA IDENTIFIER($MY_SCHEMA);
```

`MY_SCHEMA` is like a variable


# ![icon](./images/uva-icon-57x57.png) Step 1

* Create reference table

**Transcription Text:**
```
(
  'vid_106', 
  'Deploying LLMs in Production with Meta Llama 3 & Python', 
  'Cloud AI Weekly', 
  '2026-07-01', 
  18300,
  'Fine-tuning foundation models, managing token context windows, and building low-latency inference pipelines.',
  'In this episode, we build a production microservice for LLM inference using Llama3 models. We evaluate prompt engineering strategies, custom system messages, and quantization techniques to optimize throughput. We also benchmark inference latencies using GPUs vs cloud data warehouses.'
)
```

* **Table has 8 rows**
* **Transcription text, 4 categories, 2 records each**
    - Data Engineering
    - Cloud AI
    - Quantitative Finance
    - Cloud Architecture
* **2 2 2 2**

# ![icon](./images/uva-icon-57x57.png) Step 2

**Use CORTEX.COMPLETE**
* **llama3-8b**  LLM model used
* Prompt: **Classify this video transcript into EXACTLY ONE of these categories: [Data Engineering, Cloud AI, Quantitative Finance, Cloud Architecture]. Return ONLY the exact category name. Transcript:'**

## Chart
* Chart Type: Pie
* Categories: PRIMARY_DOC
* Values: # VIDEO_COUNT

**NB: SHOWS A 1 2 5 0 Distsribution**

**The output from LLM classification is NOT spot on**

**Note how vid_106 is in Data Engineering**

# ![icon](./images/uva-icon-57x57.png) Step 2B_1

**Rebuild a similar table but with different prompt**
* Prompt: **Classify this video transcript into EXACTLY ONE of these categories: [Data Engineering, Cloud AI, Quantitative Finance, Cloud Architecture]. Return ONLY the exact category name and nothing else. Transcript:**


**DO YOU SEE THE DIFFERENCE IN THE PROMPT?**

## Chart
* Chart Type: Pie
* Categories: RAW_LLM_RESPONSE
* Values: # VIDEO_COUNT

**NB: SHOWS A 2 2 4 0 Distsribution**

**The output from LLM classification is not spot on AND can be dependent on the prompt**

**Still not enough to make it catch the right classifications**

**Note how vid_106 jumped to Cloud AI**

# ![icon](./images/uva-icon-57x57.png) Step 2B_4

**Rebuild a similar table but with ORIGINAL prompt**
* Prompt: **Classify this video transcript into EXACTLY ONE of these categories: [Data Engineering, Cloud AI, Quantitative Finance, Cloud Architecture]. Return ONLY the exact category name and nothing else. Transcript:**

## Add guardrail
```sql
    CASE 
        WHEN raw_llm_response ILIKE '%Data Engineering%' THEN 'Data Engineering'
        WHEN raw_llm_response ILIKE '%Cloud AI%' THEN 'Cloud AI'
        WHEN raw_llm_response ILIKE '%Quantitative Finance%' THEN 'Quantitative Finance'
        WHEN raw_llm_response ILIKE '%Cloud Architecture%' THEN 'Cloud Architecture'
        ELSE 'Unclassified / Other'
    END AS primary_topic
```

## Chart
* Chart Type: Pie
* Categories: RAW_LLM_RESPONSE
* Values: # VIDEO_COUNT

**NB: SHOWS A 2 2 4 0 Distsribution**

**The output from LLM classification is not spot on AND can be dependent on the prompt**

**Now how vid_106 jumped back to Data Engineering**

# ![icon](./images/uva-icon-57x57.png)  Step 2C Explicit Category Rules

```sql
        SNOWFLAKE.CORTEX.COMPLETE(
            'llama3-8b', 
            'Classify this video transcript into EXACTLY ONE category using these explicit rules:
             - Quantitative Finance: Options pricing, algorithmic trading, stocks, ETFs, financial risk modeling.
             - Cloud AI: Vector embeddings, LLM inference, RAG, prompt engineering, AI microservices.
             - Cloud Architecture: Open table formats, S3 data lake storage layouts, zero-copy cloning, data sharing.
             - Data Engineering: dbt transformations, Airflow DAG orchestration, data pipelines, SQL data quality testing.
             
             Return ONLY the exact category name. Transcript: ' || transcript_text
        )
```

**NB: SHOWS A 2 2 2 2 Distsribution**

**ADDITIONALY we could have upped the model to `llama3-70B`.  More expensive, but more accurate**

# ![icon](./images/uva-icon-57x57.png) Recap

## All the topics we covered
* Using the linux command line
* Provisioning your own AWS EC2 Ubuntu server
* Hooking up git keys
* Set up a youtube transcript retriever
    - equipped with a proxy so you can use it from AWS
* Set up an id cleaner
* Used an external LLM, Gemini, to extract data from your transcripts
* Pushed that data in json format to snowflake
* Packaged all that up in a Docker container
* Used git in snowflake
* Set up dbt tests and models for your data
* Transform data using Snowflake Cortex


**That's a LOT!**

# ![icon](./images/uva-icon-57x57.png)  Key Takeaways

## **Focus on PROCESS not the tools**
True, some tools are important, which is why we used dbt/snowflake.  But the **process** is what you **fall to when things get busy, stressful!**

## Recommendations
* **Internalize the Automation**
    - **makefiles**
    - **pytest** (and pylint)
* **Incorporate testing into your process**  The earlier you find a bug, the more time you save
* **Make it a habit to think about how you organize your code**
    - **OOP**
    - **Linux pipeline concept:  The output one one app becomes the input for the next.**
    - **Design Patterns**
* **Incorporate AI into your learnings, projects.  I can't think of any company who is going 'old school'**
    - Note how incredibly useful and time saving the tech can be, think of doing what gemini llm did for us, but in a script.
    - Note how brittle and undetermistic it can be, think of today's CORTEX example.
    - Get into the habit of using dbt/dbt-expectation to put rails on your data
* **Practice the end to end with your projects, build a full pipeline**
    - Not because you are going to build them at work, but so that you can unblock yourself when resources are tight.
* **Play with Docker**
    - Literally go to youtube and search for "docker build data science lab"...

**AI is a game change, which makes the process emphasis more pronounced.  Let AI handle the syntax... you handle the higher levels**

# ![icon](./images/uva-icon-57x57.png)  


**THANK YOU FOR BEING IN THE CLASS!!**

## Assignments this week

* **PLEASE** try/read the lab end to end.
* For those of you who have missing labs, or got stuck, turn it in as **EXTRA CREDIT**
* There is one more **Quiz**

Find me on LinkedIn [https://www.linkedin.com/in/efrain-olivares-62173a5/](https://www.linkedin.com/in/efrain-olivares-62173a5/)